## 1. Imports and Setup

In [ ]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, replace
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon
from shapely.affinity import translate
import networkx as nx
from sklearn.neighbors import NearestNeighbors

In [ ]:
@dataclass
class MatchCaseConfig:
    name: str
    base_similarity_weights: Dict[str, float]
    scoring_weights: Dict[str, float]
    similarity_threshold: float
    min_iou: float = 0.0
    min_overlap_prev: float = 0.0
    min_overlap_curr: float = 0.0
    max_centroid_dist: Optional[float] = None
    mutual_best: bool = False
    allow_multiple: bool = False
    max_edges_per_prev: Optional[int] = None
    max_edges_per_curr: Optional[int] = None


class TreeTrackingGraph:
    """
    Tree crown tracker across orthomosaics using a directed graph with ultra-relaxed parameters.
    
    Supports automatic file discovery, image extraction, spatial alignment, and configurable matching.
    """
    
    def __init__(
        self,
        crown_dir: Optional[str] = None,
        ortho_dir: Optional[str] = None,
        output_dir: str = '../../output',
        simplify_tol: float = 1.0,
        max_crowns_preview: int = 200,
        auto_discover: bool = True
    ) -> None:
        self.output_dir = output_dir
        self.simplify_tol = simplify_tol
        self.max_crowns_preview = max_crowns_preview
        self.crown_dir = crown_dir or self._resolve_dir('input/input_crowns', '../../input/input_crowns')
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        
        self.file_pairs: List[Tuple[str, Optional[str]]] = []
        self.om_ids: List[int] = []
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}
        self.G: nx.DiGraph = nx.DiGraph()
        
        # Use ultra-relaxed configs by default
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        if auto_discover:
            self.discover_files()
    
    @staticmethod
    def _resolve_dir(root_rel: str, nb_rel: str) -> str:
        """Resolve directory path from multiple possible locations."""
        candidates = [
            os.path.abspath(os.path.join(os.getcwd(), root_rel)),
            os.path.abspath(os.path.join(os.getcwd(), nb_rel)),
        ]
        for p in candidates:
            if os.path.isdir(p):
                return p
        raise FileNotFoundError(f"Could not resolve directory for {root_rel}. Tried: {candidates}")
    
    @staticmethod
    def _extract_numeric_id(name: str) -> Optional[int]:
        """Extract numeric ID from filename."""
        m = re.search(r"(\d+)", os.path.basename(name))
        return int(m.group(1)) if m else None
    
    def discover_files(self) -> None:
        """Automatically discover and pair crown GeoPackages with orthomosaics."""
        crown_files = [
            os.path.join(self.crown_dir, f) 
            for f in os.listdir(self.crown_dir) 
            if f.lower().endswith('.gpkg')
        ]
        ortho_files = [
            os.path.join(self.ortho_dir, f) 
            for f in os.listdir(self.ortho_dir) 
            if f.lower().endswith('.tif')
        ] if os.path.exists(self.ortho_dir) else []
        
        if not crown_files:
            raise FileNotFoundError(f"No .gpkg crown files found in {self.crown_dir}")
        
        # Group files by numeric ID
        crowns_by_id = {}
        for cf in crown_files:
            cid = self._extract_numeric_id(cf)
            crowns_by_id[cid if cid is not None else cf] = cf
        
        orthos_by_id = {}
        for of in ortho_files:
            oid = self._extract_numeric_id(of)
            orthos_by_id[oid if oid is not None else of] = of
        
        # Find matching numeric IDs
        numeric_ids = sorted(
            set(k for k in crowns_by_id.keys() if isinstance(k, int)) & 
            set(k for k in orthos_by_id.keys() if isinstance(k, int))
        )
        
        # Pair files
        file_pairs: List[Tuple[str, Optional[str]]] = []
        if numeric_ids:
            for nid in numeric_ids:
                file_pairs.append((crowns_by_id[nid], orthos_by_id.get(nid)))
            crown_only = sorted(
                k for k in crowns_by_id.keys() 
                if isinstance(k, int) and k not in numeric_ids
            )
            for nid in crown_only:
                file_pairs.append((crowns_by_id[nid], None))
        else:
            crown_files_sorted = sorted(crown_files)
            ortho_files_sorted = sorted(ortho_files)
            for i, cf in enumerate(crown_files_sorted):
                of = ortho_files_sorted[i] if i < len(ortho_files_sorted) else None
                file_pairs.append((cf, of))
        
        # Extract OM IDs
        om_ids: List[int] = []
        for cf, _ in file_pairs:
            cid = self._extract_numeric_id(cf)
            om_ids.append(cid if cid is not None else len(om_ids) + 1)
        
        # Sort by OM ID
        pairs_with_id = sorted(
            [(oid, cf, of) for oid, (cf, of) in zip(om_ids, file_pairs)],
            key=lambda x: x[0]
        )
        self.file_pairs = [(cf, of) for _, cf, of in pairs_with_id]
        self.om_ids = [oid for oid, _, _ in pairs_with_id]
    
    def load_data(
        self,
        load_images: bool = False,
        align: bool = False,
        reference_om_id: Optional[int] = None
    ) -> None:
        """
        Load crown data from GeoPackages and optionally extract image patches.
        
        CRITICAL: Images are extracted from ORIGINAL geometries BEFORE alignment.
        This ensures image-geometry correspondence is preserved.
        
        Args:
            load_images: If True, extract RGB image patches from orthomosaics for each crown
            align: If True, align all OMs to reference OM using translational shift
            reference_om_id: OM ID to use as alignment reference (default: first OM)
        """
        self.crowns_gdfs.clear()
        self.crown_attrs.clear()
        self.crown_images.clear()
        
        print(f"\n{'='*70}")
        print("LOADING DATA")
        print(f"{'='*70}")
        
        # Step 1: Load geometries and extract images from ORIGINAL positions
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            print(f"\nOM{om_id}: Loading {os.path.basename(crown_file)}")
            gdf = gpd.read_file(crown_file)
            self.crowns_gdfs[om_id] = gdf
            
            # Extract images from ORIGINAL geometries BEFORE any shifting
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    patches: List[Optional[np.ndarray]] = []
                    for idx, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]
                        try:
                            out_image, _ = mask(src, geom, crop=True)
                            img_patch = np.moveaxis(out_image, 0, -1)  # (C, H, W) -> (H, W, C)
                        except Exception:
                            img_patch = None
                        patches.append(img_patch)
                self.crown_images[om_id] = patches
                print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
            
            # Compute attributes from original geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        print(f"\n{'='*70}")
        
        # Step 2: Apply alignment if requested (geometries shift, images stay with original crowns)
        if align:
            print(f"\n{'='*70}")
            print("APPLYING SPATIAL ALIGNMENT")
            print(f"{'='*70}")
            self.alignment_shifts = self.align_crowns()
            print(f"{'='*70}\n")
    
    def align_crowns(self, threshold: float = 10.0) -> Dict[int, Tuple[float, float]]:
        """
        Align crowns consecutively using nearest neighbors and median translation.
        
        Based on the algorithm from crown_tracking_real_data_demo.ipynb.
        
        Args:
            threshold: Maximum distance for considering a nearest neighbor match (meters)
        
        Returns:
            Dictionary mapping om_id -> (dx, dy) shift applied
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        
        shifts = {self.om_ids[0]: (0.0, 0.0)}  # Reference doesn't shift
        aligned = [self.crowns_gdfs[self.om_ids[0]].copy()]
        
        for t in range(1, len(self.om_ids)):
            om_id = self.om_ids[t]
            ref = aligned[-1]
            curr = self.crowns_gdfs[om_id].copy()
            
            centroids_ref = np.array([[g.centroid.x, g.centroid.y] for g in ref.geometry])
            centroids_curr = np.array([[g.centroid.x, g.centroid.y] for g in curr.geometry])
            
            nn = NearestNeighbors(n_neighbors=1).fit(centroids_ref)
            distances, indices = nn.kneighbors(centroids_curr)
            matched = [(indices[i][0], i) for i in range(len(distances)) if distances[i][0] < threshold]
            
            if len(matched) < 3:
                print(f'Not enough matches for OM{om_id}, skipping alignment.')
                shifts[om_id] = (0.0, 0.0)
                aligned.append(curr)
                continue
            
            shifts_array = np.array([centroids_ref[i] - centroids_curr[j] for i, j in matched])
            residuals = np.linalg.norm(shifts_array, axis=1)
            inliers = residuals < np.percentile(residuals, 90)
            dx, dy = np.median(shifts_array[inliers], axis=0)
            
            print(f'OM{om_id}: Applying median translation dx={dx:.2f}, dy={dy:.2f}')
            curr['geometry'] = curr['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
            aligned.append(curr)
            shifts[om_id] = (dx, dy)
        
        # Update the crowns_gdfs with aligned versions
        for i, om_id in enumerate(self.om_ids):
            self.crowns_gdfs[om_id] = aligned[i]
            
            # Recompute attributes with shifted geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in self.crowns_gdfs[om_id].iterrows()
            ]
        
        return shifts
    
    @staticmethod
    def _compute_crown_attributes(geometry) -> Dict[str, Any]:
        """Compute geometric attributes for a crown polygon."""
        centroid = geometry.centroid
        area = geometry.area
        perimeter = geometry.length
        bounds = geometry.bounds
        compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        try:
            min_rect = geometry.minimum_rotated_rectangle
            coords = list(min_rect.exterior.coords)
            side1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
            side2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))
            major_axis = max(side1, side2)
            minor_axis = min(side1, side2)
            eccentricity = minor_axis / major_axis if major_axis > 0 else 1
        except Exception:
            eccentricity = 1
        
        aspect_ratio = (bounds[3] - bounds[1]) / (bounds[2] - bounds[0]) if bounds[2] != bounds[0] else 1
        
        return {
            'geometry': geometry,
            'centroid': centroid,
            'area': area,
            'perimeter': perimeter,
            'compactness': compactness,
            'eccentricity': eccentricity,
            'aspect_ratio': aspect_ratio,
            'bounds': bounds,
        }
    
    def _ultra_relaxed_case_configs(self) -> Dict[str, MatchCaseConfig]:
        """
        ULTRA RELAXED matching parameters - prioritizes spatial proximity over shape similarity.
        
        Designed to bridge difficult OM transitions with minimal overlap or shape changes.
        """
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={
                    'spatial': 0.50,  # Strong emphasis on proximity
                    'area': 0.15,
                    'shape': 0.05,    # De-emphasize shape
                    'iou': 0.30
                },
                scoring_weights={
                    'base': 0.60, 
                    'iou': 0.10, 
                    'overlap_prev': 0.05, 
                    'overlap_curr': 0.05, 
                    'centroid': 0.20
                },
                similarity_threshold=0.25,
                min_iou=0.01,
                min_overlap_prev=0.10,
                min_overlap_curr=0.10,
                max_centroid_dist=50.0,
                mutual_best=False,
                allow_multiple=False,
                max_edges_per_prev=3,
                max_edges_per_curr=3,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={
                    'spatial': 0.50,
                    'area': 0.20,
                    'shape': 0.10,
                    'iou': 0.20
                },
                scoring_weights={
                    'base': 0.40, 
                    'overlap_prev': 0.30, 
                    'overlap_curr': 0.30
                },
                similarity_threshold=0.25,
                min_iou=0.0,
                min_overlap_prev=0.25,
                min_overlap_curr=0.25,
                max_centroid_dist=150.0,
                mutual_best=False,
                allow_multiple=False,
                max_edges_per_prev=1,
                max_edges_per_curr=1,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={
                    'spatial': 0.85,  # Almost purely spatial
                    'area': 0.10,
                    'shape': 0.03,
                    'iou': 0.02
                },
                scoring_weights={
                    'base': 0.70, 
                    'centroid': 0.30
                },
                similarity_threshold=0.20,
                min_iou=0.0,
                min_overlap_prev=0.05,
                min_overlap_curr=0.05,
                max_centroid_dist=200.0,
                mutual_best=False,
                allow_multiple=False,
                max_edges_per_prev=1,
                max_edges_per_curr=1,
            ),
        }
    
    @staticmethod
    def _compute_iou(g1, g2) -> float:
        """Compute intersection over union for two geometries."""
        try:
            intersection = g1.intersection(g2).area
            union = g1.union(g2).area
        except Exception:
            intersection = 0.0
            union = g1.area + g2.area
        return intersection / union if union > 0 else 0.0
    
    def _weighted_similarity(
        self,
        a1: Dict[str, Any],
        a2: Dict[str, Any],
        weights: Optional[Dict[str, float]] = None,
        max_dist: float = 100.0
    ) -> Tuple[float, Dict[str, float]]:
        """Compute weighted similarity score between two crown attributes."""
        if weights is None:
            weights = {'spatial': 0.4, 'area': 0.2, 'shape': 0.2, 'iou': 0.2}
        
        # Spatial similarity
        centroid_dist = a1['centroid'].distance(a2['centroid'])
        spatial_sim = max(0.0, 1.0 - (centroid_dist / max_dist))
        
        # Area similarity
        area_sim = min(a1['area'], a2['area']) / max(a1['area'], a2['area']) if max(a1['area'], a2['area']) > 0 else 0.0
        
        # Shape similarity
        compactness_sim = 1.0 - abs(a1['compactness'] - a2['compactness'])
        eccentricity_sim = 1.0 - abs(a1['eccentricity'] - a2['eccentricity'])
        shape_sim = (compactness_sim + eccentricity_sim) / 2.0
        
        # IoU similarity
        iou_sim = self._compute_iou(a1['geometry'], a2['geometry'])
        
        # Weighted total
        total = (
            weights.get('spatial', 0.0) * spatial_sim +
            weights.get('area', 0.0) * area_sim +
            weights.get('shape', 0.0) * shape_sim +
            weights.get('iou', 0.0) * iou_sim
        )
        
        return total, {
            'spatial': float(spatial_sim),
            'area': float(area_sim),
            'shape': float(shape_sim),
            'iou': float(iou_sim),
            'total': float(total)
        }
    
    def _compute_pair_metrics(
        self,
        prev_attrs: Dict[str, Any],
        curr_attrs: Dict[str, Any],
        max_dist: float
    ) -> Dict[str, float]:
        """Compute comprehensive metrics for a crown pair."""
        prev_geom = prev_attrs['geometry']
        curr_geom = curr_attrs['geometry']
        
        try:
            intersection_area = prev_geom.intersection(curr_geom).area
        except Exception:
            intersection_area = 0.0
        
        try:
            union_area = prev_geom.union(curr_geom).area
        except Exception:
            union_area = prev_attrs['area'] + curr_attrs['area'] - intersection_area
        
        prev_area = prev_attrs['area'] if prev_attrs['area'] > 0 else 1e-6
        curr_area = curr_attrs['area'] if curr_attrs['area'] > 0 else 1e-6
        
        overlap_prev = intersection_area / prev_area
        overlap_curr = intersection_area / curr_area
        iou = intersection_area / union_area if union_area > 0 else 0.0
        
        centroid_dist = prev_attrs['centroid'].distance(curr_attrs['centroid'])
        base_similarity, parts = self._weighted_similarity(prev_attrs, curr_attrs, max_dist=max_dist)
        
        prev_radius = np.sqrt(prev_area / np.pi)
        curr_radius = np.sqrt(curr_area / np.pi)
        mean_radius = max((prev_radius + curr_radius) / 2.0, 1e-3)
        
        area_ratio = curr_area / prev_area if prev_area > 0 else np.inf
        if not np.isfinite(area_ratio) or area_ratio <= 0:
            balanced_area_ratio = 0.0
        else:
            balanced_area_ratio = area_ratio if area_ratio <= 1 else 1 / area_ratio
        
        try:
            prev_contains_curr = prev_geom.buffer(0).contains(curr_geom)
        except Exception:
            prev_contains_curr = False
        
        try:
            curr_contains_prev = curr_geom.buffer(0).contains(prev_geom)
        except Exception:
            curr_contains_prev = False
        
        return {
            'intersection_area': float(intersection_area),
            'union_area': float(union_area),
            'overlap_prev': float(overlap_prev),
            'overlap_curr': float(overlap_curr),
            'iou': float(iou),
            'centroid_dist': float(centroid_dist),
            'base_similarity': float(base_similarity),
            'spatial_similarity': float(parts['spatial']),
            'area_similarity': float(parts['area']),
            'shape_similarity': float(parts['shape']),
            'mean_radius': float(mean_radius),
            'area_ratio': float(area_ratio if np.isfinite(area_ratio) else 0.0),
            'balanced_area_ratio': float(balanced_area_ratio),
            'prev_contains_curr': bool(prev_contains_curr),
            'curr_contains_prev': bool(curr_contains_prev),
        }
    
    def _classify_match_case(
        self,
        prev_node: Tuple[int, int],
        curr_node: Tuple[int, int],
        features: Dict[str, float],
        prev_overlap_counts: Dict[Tuple[int, int], int],
        curr_overlap_counts: Dict[Tuple[int, int], int],
        overlap_gate: float
    ) -> str:
        """Classify the type of match between two crowns."""
        # Containment case
        if features['prev_contains_curr'] or features['curr_contains_prev']:
            return 'containment'
        
        # Extract features
        overlap_prev = features['overlap_prev']
        overlap_curr = features['overlap_curr']
        iou = features['iou']
        prev_count = prev_overlap_counts.get(prev_node, 0)
        curr_count = curr_overlap_counts.get(curr_node, 0)
        centroid_dist = features['centroid_dist']
        
        # One-to-one: Use overlap_gate parameter dynamically
        min_overlap_for_one_to_one = max(overlap_gate, 0.30)
        min_iou_for_one_to_one = max(overlap_gate / 2, 0.10)
        
        if (prev_count == 1 and curr_count == 1 and 
            overlap_prev >= min_overlap_for_one_to_one and 
            overlap_curr >= min_overlap_for_one_to_one and 
            iou >= min_iou_for_one_to_one):
            return 'one_to_one'
        
        # Nearby: Primarily spatial proximity with minimal overlap requirement
        if centroid_dist < 30.0 and (overlap_prev >= overlap_gate or overlap_curr >= overlap_gate):
            return 'nearby'
        
        return 'none'
    
    def _score_candidate(
        self,
        base_similarity: float,
        similarity_parts: Dict[str, float],
        features: Dict[str, float],
        config: MatchCaseConfig
    ) -> float:
        """Score a candidate match using weighted components."""
        centroid_factor = 1.0 - min(1.0, features['centroid_dist'] / (features['mean_radius'] * 3.0))
        
        components = {
            'base': base_similarity,
            'spatial': similarity_parts.get('spatial', 0.0),
            'area': similarity_parts.get('area', 0.0),
            'shape': similarity_parts.get('shape', 0.0),
            'iou': features['iou'],
            'overlap_prev': features['overlap_prev'],
            'overlap_curr': features['overlap_curr'],
            'centroid': max(0.0, centroid_factor),
            'area_balance': features.get('balanced_area_ratio', 0.0),
        }
        
        score = 0.0
        for key, weight in config.scoring_weights.items():
            score += weight * components.get(key, 0.0)
        
        return score
    
    def _select_candidates_by_case(
        self,
        candidates: List[Dict[str, Any]],
        configs: Dict[str, MatchCaseConfig],
        case_order: List[str],
        max_dist: float
    ) -> List[Dict[str, Any]]:
        """Select best candidates for each case in priority order."""
        selected: List[Dict[str, Any]] = []
        used_prev: Dict[Tuple[int, int], int] = defaultdict(int)
        used_curr: Dict[Tuple[int, int], int] = defaultdict(int)
        
        for case_name in case_order:
            config = configs.get(case_name)
            if not config:
                continue
            
            case_candidates = [cand for cand in candidates if cand['case'] == case_name]
            if not case_candidates:
                continue
            
            # Enrich candidates with scores
            enriched: List[Dict[str, Any]] = []
            for cand in case_candidates:
                prev_attrs = cand['prev_attrs']
                curr_attrs = cand['curr_attrs']
                features = cand['features']
                
                # Apply filters
                if config.max_centroid_dist is not None and features['centroid_dist'] > config.max_centroid_dist:
                    continue
                if features['iou'] < config.min_iou:
                    continue
                if features['overlap_prev'] < config.min_overlap_prev:
                    continue
                if features['overlap_curr'] < config.min_overlap_curr:
                    continue
                
                # Compute score
                base_similarity, parts = self._weighted_similarity(
                    prev_attrs, curr_attrs,
                    weights=config.base_similarity_weights,
                    max_dist=max_dist
                )
                score = self._score_candidate(base_similarity, parts, features, config)
                
                if score < config.similarity_threshold:
                    continue
                
                cand['base_similarity'] = float(base_similarity)
                cand['similarity_parts'] = {k: float(v) for k, v in parts.items()}
                cand['score'] = float(score)
                enriched.append(cand)
            
            if not enriched:
                continue
            
            # Select candidates
            if config.mutual_best:
                best_prev: Dict[Tuple[int, int], Dict[str, Any]] = {}
                best_curr: Dict[Tuple[int, int], Dict[str, Any]] = {}
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if used_prev.get(prev_node, 0) and not config.allow_multiple:
                        continue
                    if used_curr.get(curr_node, 0) and not config.allow_multiple:
                        continue
                    
                    if cand['score'] < config.similarity_threshold:
                        continue
                    
                    if prev_node not in best_prev or cand['score'] > best_prev[prev_node]['score']:
                        best_prev[prev_node] = cand
                    if curr_node not in best_curr or cand['score'] > best_curr[curr_node]['score']:
                        best_curr[curr_node] = cand
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if best_prev.get(prev_node) is cand and best_curr.get(curr_node) is cand:
                        if not config.allow_multiple:
                            if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                                continue
                        
                        if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                            continue
                        if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                            continue
                        
                        selected.append(cand)
                        used_prev[prev_node] += 1
                        used_curr[curr_node] += 1
            else:
                enriched.sort(key=lambda c: c['score'], reverse=True)
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                        continue
                    if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                        continue
                    
                    selected.append(cand)
                    used_prev[prev_node] += 1
                    used_curr[curr_node] += 1
        
        return selected
    
    def reset_graph(self) -> None:
        """Clear the graph."""
        self.G = nx.DiGraph()
    
    def build_graph_conditional(
        self,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        base_max_dist: float = 200.0,
        overlap_gate: float = 0.05,
        min_base_similarity: float = 0.05,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None
    ) -> None:
        """
        Build tracking graph using conditional case-based matching.
        
        Args:
            case_configs: Dictionary of case configurations (default: ultra-relaxed)
            case_order: Priority order for cases (default: ['one_to_one', 'containment', 'nearby'])
            base_max_dist: Maximum centroid distance for candidate matches (meters)
            overlap_gate: Minimum overlap fraction to count as overlapping
            min_base_similarity: Minimum base similarity for candidate filtering
            max_candidates_per_prev: Maximum candidates to consider per previous crown
            max_candidates_per_curr: Maximum candidates to consider per current crown
        """
        if not self.crowns_gdfs:
            self.load_data(load_images=False)
        
        self.reset_graph()
        
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        
        self.last_case_counts = {}
        self.last_selected_counts = {name: 0 for name in configs.keys()}
        
        print(f"\n{'='*70}")
        print("BUILDING TRACKING GRAPH")
        print(f"{'='*70}")
        print(f"Parameters:")
        print(f"  base_max_dist: {base_max_dist}m")
        print(f"  overlap_gate: {overlap_gate}")
        print(f"  min_base_similarity: {min_base_similarity}")
        print(f"  case_order: {', '.join(order)}")
        print()
        
        # Add all nodes
        for idx in range(len(self.om_ids)):
            om_id = self.om_ids[idx]
            gdf = self.crowns_gdfs[om_id]
            
            for crown_id, row in gdf.iterrows():
                attrs = self.crown_attrs[om_id][crown_id]
                self.G.add_node((om_id, crown_id), **attrs)
            
            if idx == 0:
                continue
            
            # Build edges between consecutive OMs
            prev_om = self.om_ids[idx - 1]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(om_id, j) for j in range(len(gdf))]
            
            print(f"OM{prev_om} → OM{om_id}: {len(prev_nodes)} × {len(curr_nodes)} potential pairs")
            
            # Generate candidates
            candidates: List[Dict[str, Any]] = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            for prev_node in prev_nodes:
                prev_attrs = self.G.nodes[prev_node]
                
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[om_id][curr_node[1]]
                    
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    
                    # Early filtering
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    if features['base_similarity'] < min_base_similarity and features['iou'] < overlap_gate:
                        continue
                    
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            if not candidates:
                print(f"  No candidates found")
                continue
            
            # Classify candidates
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate
                )
            
            candidates = [cand for cand in candidates if cand['case'] != 'none']
            
            if not candidates:
                print(f"  No valid cases found")
                continue
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_prev[cand['prev_node']].append(cand)
                
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (c['features']['base_similarity'], c['features']['iou']), reverse=True)
                    trimmed.extend(group[:max_candidates_per_prev])
                candidates = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_curr[cand['curr_node']].append(cand)
                
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (c['features']['base_similarity'], c['features']['iou']), reverse=True)
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                candidates = trimmed_curr
            
            # Count candidates by case
            case_counts = defaultdict(int)
            for cand in candidates:
                case_counts[cand['case']] += 1
            
            for case_name, count in case_counts.items():
                self.last_case_counts[case_name] = self.last_case_counts.get(case_name, 0) + count
            
            print(f"  Candidates by case: {dict(case_counts)}")
            
            # Select best candidates
            selected = self._select_candidates_by_case(candidates, configs, order, base_max_dist)
            
            # Add edges to graph
            for cand in selected:
                case_name = cand['case']
                features = cand['features']
                similarity_parts = cand.get('similarity_parts', {})
                
                self.G.add_edge(
                    cand['prev_node'],
                    cand['curr_node'],
                    similarity=float(cand.get('score', features['base_similarity'])),
                    method='conditional',
                    case=case_name,
                    overlap_prev=float(features['overlap_prev']),
                    overlap_curr=float(features['overlap_curr']),
                    iou=float(features['iou']),
                    centroid_distance=float(features['centroid_dist']),
                    base_similarity=float(cand.get('base_similarity', features['base_similarity'])),
                    spatial_similarity=float(similarity_parts.get('spatial', features['spatial_similarity'])),
                    area_similarity=float(similarity_parts.get('area', features['area_similarity'])),
                    shape_similarity=float(similarity_parts.get('shape', features['shape_similarity']))
                )
                
                self.last_selected_counts[case_name] = self.last_selected_counts.get(case_name, 0) + 1
            
            print(f"  Selected: {len(selected)} edges")
            print()
        
        print(f"{'='*70}")
        print(f"Graph built: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges")
        print(f"{'='*70}\n")
    
    def _extract_all_chains(self) -> List[List[Tuple[int, int]]]:
        """Extract all tracking chains from the graph."""
        visited = set()
        chains: List[List[Tuple[int, int]]] = []
        
        # Start from nodes with no predecessors
        chain_starts = [n for n in self.G.nodes if not list(self.G.predecessors(n))]
        
        for start_node in chain_starts:
            if start_node in visited:
                continue
            
            chain = self._greedy_chain(start_node)
            chains.append(chain)
            visited.update(chain)
        
        # Handle remaining nodes (cycles or isolated)
        remaining = set(self.G.nodes) - visited
        for node in remaining:
            chains.append([node])
        
        return chains
    
    def _greedy_chain(self, start_node: Tuple[int, int]) -> List[Tuple[int, int]]:
        """Extract a single chain starting from a given node."""
        chain = [start_node]
        current = start_node
        
        while True:
            successors = list(self.G.successors(current))
            if not successors:
                break
            
            if len(successors) > 1:
                # Choose best successor
                best_successor = max(
                    successors,
                    key=lambda n: self.G[current][n].get('similarity', 0.0)
                )
                chain.append(best_successor)
                current = best_successor
            else:
                chain.append(successors[0])
                current = successors[0]
        
        return chain
    
    def quality_report(self) -> Tuple[str, Dict[str, Any]]:
        """Generate quality metrics and report."""
        G = self.G
        om_ids = self.om_ids
        
        metrics: Dict[str, Any] = {
            'total_trees_detected': G.number_of_nodes(),
            'total_edges': G.number_of_edges(),
            'total_possible_matches': 0,
            'successful_matches': 0,
            'match_rate_by_om_pair': {},
            'chain_length_distribution': {},
            'average_chain_length': 0,
            'median_chain_length': 0,
            'max_chain_length': 0,
        }
        
        # Chain analysis
        chains = self._extract_all_chains()
        chain_lengths = [len(chain) for chain in chains]
        
        if chain_lengths:
            metrics['average_chain_length'] = float(np.mean(chain_lengths))
            metrics['median_chain_length'] = float(np.median(chain_lengths))
            metrics['max_chain_length'] = int(max(chain_lengths))
            
            for length in chain_lengths:
                metrics['chain_length_distribution'][int(length)] = metrics['chain_length_distribution'].get(int(length), 0) + 1
        
        # Match rate by OM pair
        for i in range(len(om_ids) - 1):
            om1, om2 = om_ids[i], om_ids[i + 1]
            om1_nodes = [n for n in G.nodes if n[0] == om1]
            om2_nodes = [n for n in G.nodes if n[0] == om2]
            
            matches = sum(1 for u, v in G.edges() if u[0] == om1 and v[0] == om2)
            possible_matches = min(len(om1_nodes), len(om2_nodes))
            match_rate = matches / possible_matches if possible_matches > 0 else 0.0
            
            metrics['match_rate_by_om_pair'][f"{om1}->{om2}"] = {
                'matches': matches,
                'possible': possible_matches,
                'rate': float(match_rate),
            }
            
            metrics['total_possible_matches'] += possible_matches
            metrics['successful_matches'] += matches
        
        metrics['overall_match_rate'] = (
            metrics['successful_matches'] / metrics['total_possible_matches']
            if metrics['total_possible_matches'] > 0 else 0.0
        )
        
        # Generate report
        report = [
            "# Tree Tracking Quality Assessment Report",
            f"Total Trees Detected: {metrics['total_trees_detected']}",
            f"Total Tracking Edges: {metrics['total_edges']}",
            f"Overall Match Rate: {metrics['overall_match_rate']:.3f}",
            f"Average Chain Length: {metrics.get('average_chain_length', 0):.2f}",
            f"Maximum Chain Length: {metrics.get('max_chain_length', 0)}",
            "\nMatch Rates by Orthomosaic Pair:",
        ]
        
        for pair, data in metrics['match_rate_by_om_pair'].items():
            report.append(f"- {pair}: {data['matches']}/{data['possible']} ({data['rate']:.3f})")
        
        report.append("\nChain Length Distribution:")
        for length, count in sorted(metrics['chain_length_distribution'].items()):
            report.append(f"- Length {length}: {count} trees")
        
        if self.last_selected_counts:
            report.append("\nEdge selection by case:")
            for case_name, count in sorted(self.last_selected_counts.items(), key=lambda kv: (-kv[1], kv[0])):
                total_candidates = self.last_case_counts.get(case_name, 0)
                if total_candidates:
                    ratio = count / total_candidates
                    report.append(f"- {case_name}: {count} / {total_candidates} ({ratio:.2f})")
                else:
                    report.append(f"- {case_name}: {count}")
        
        return "\n".join(report), metrics
    
    def get_matching_chain(self, start_om_id: int, crown_id: int) -> List[Tuple[int, int]]:
        """Get the tracking chain starting from a specific crown."""
        node = (start_om_id, crown_id)
        if node not in self.G:
            raise ValueError(f"Node {(start_om_id, crown_id)} not in graph. Build the graph first.")
        return self._greedy_chain(node)
    
    def ensure_output_dir(self) -> None:
        """Create output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)
    
    def save_text(self, text: str, filename: str) -> str:
        """Save text to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            f.write(text)
        return path
    
    def save_json(self, data: Dict[str, Any], filename: str) -> str:
        """Save JSON data to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        return path

print("✓ TreeTrackingGraph class defined with ultra-relaxed parameters")

## 2. TreeTrackingGraph Class

The main class for tree crown tracking across orthomosaics.

## 3. Visualization Functions

Functions for visualizing tracking chains with polygons and extracted RGB images.

In [ ]:
def visualize_chain_with_extracted_images(chain, aligned_gdfs, crown_images_dict, tracker, title="Chain Example", save_path=None, show=True, close_fig=True, dpi=150):
    """
    Visualize chain with both crown polygons AND extracted RGB images.
    Optional saving for PPT-friendly reuse.

    Args:
        chain: List of (om_id, crown_idx) tuples
        aligned_gdfs: Dictionary of aligned GeoDataFrames by OM ID
        crown_images_dict: Dictionary of crown images by OM ID (from tracker.crown_images)
        tracker: TreeTrackingGraph instance
        title: Plot title
        save_path: Optional path to save the figure
        show: Whether to display the plot
        close_fig: Close the figure after saving/showing to free memory
        dpi: Resolution when saving
    """
    chain_length = len(chain)
    
    # Create figure with 2 rows: polygons on top, images on bottom
    fig = plt.figure(figsize=(5*chain_length, 10))
    
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    
    # Plot each crown in the chain
    for idx, (om_idx, crown_idx) in enumerate(chain):
        gdf = aligned_gdfs[om_idx]
        crown = gdf.iloc[crown_idx]
        
        # Get edge info if not last crown
        edge_info = ""
        if idx < chain_length - 1:
            next_node = chain[idx + 1]
            if tracker.G.has_edge((om_idx, crown_idx), next_node):
                edge_data = tracker.G.edges[(om_idx, crown_idx), next_node]
                edge_info = f" → OM{next_node[0]}[{next_node[1]}]: case={edge_data['case']}, sim={edge_data['similarity']:.2f}, iou={edge_data.get('iou', 0):.2f}"
        
        # Print crown info
        in_deg = tracker.G.in_degree((om_idx, crown_idx))
        out_deg = tracker.G.out_degree((om_idx, crown_idx))
        print(f"  OM{om_idx}[{crown_idx}]: in_deg={in_deg}, out_deg={out_deg}, area={crown.geometry.area:.1f}m²{edge_info}")
        
        # ROW 1: Crown polygon
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        
        # Get bounds and plot
        minx, miny, maxx, maxy = crown.geometry.bounds
        margin = max((maxx - minx), (maxy - miny)) * 0.3
        
        # Plot other crowns in gray
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        
        # Plot this crown highlighted
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor=plt.cm.tab10(om_idx-1),
            edgecolor='black',
            linewidth=2,
            alpha=0.7
        )
        
        # Plot centroid
        centroid = crown.geometry.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                    markeredgecolor='black', markeredgewidth=1.5)
        
        # Arrow to next crown
        if idx < chain_length - 1:
            ax_poly.annotate('', xy=(maxx + margin*0.5, (miny+maxy)/2),
                            xytext=(maxx + margin*0.2, (miny+maxy)/2),
                            arrowprops=dict(arrowstyle='->', lw=3, color='red'))
        
        ax_poly.set_xlim(minx - margin, maxx + margin)
        ax_poly.set_ylim(miny - margin, maxy + margin)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(
            f"OM{om_idx} - Crown {crown_idx}\nArea: {crown.geometry.area:.1f}m²\nIn:{in_deg} Out:{out_deg}",
            fontsize=10, fontweight='bold'
        )
        ax_poly.set_xlabel("X (m)", fontsize=9)
        ax_poly.set_ylabel("Y (m)", fontsize=9)
        ax_poly.grid(True, alpha=0.3)
        
        # Add edge case label
        if edge_info:
            edge_data = tracker.G.edges[(om_idx, crown_idx), chain[idx + 1]]
            bbox_color = {
                'one_to_one': 'wheat',
                'containment': 'lightblue',
                'nearby': 'lightcoral'
            }.get(edge_data['case'], 'white')
            ax_poly.text(
                0.95, 0.95,
                f"→ {edge_data['case']}\nSim: {edge_data['similarity']:.2f}\nIoU: {edge_data.get('iou', 0):.2f}",
                transform=ax_poly.transAxes, fontsize=8,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor=bbox_color, alpha=0.8)
            )
        
        # ROW 2: Extracted RGB image
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        
        rgb_image = crown_images_dict[om_idx][crown_idx]
        if rgb_image is not None:
            ax_img.imshow(rgb_image)
            ax_img.set_title(
                f"Extracted Tree Image\n{rgb_image.shape[0]}×{rgb_image.shape[1]} px",
                fontsize=9
            )
        else:
            ax_img.text(0.5, 0.5, 'Image not available',
                       ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        
        ax_img.axis('off')
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    if close_fig:
        plt.close(fig)
    print()
    return save_path


def categorize_chains(tracker):
    """
    Categorize all chains from the tracker.
    
    Returns:
        Dictionary with categorized chains
    """
    all_chains = tracker._extract_all_chains()
    max_chain_length = len(tracker.om_ids)
    
    full_chains_width_1 = []      # Full length chains with no branching
    full_chains_branching = []    # Full length chains with branching
    partial_chains_long = []      # Length 3-4
    partial_chains_short = []     # Length 2
    
    for chain in all_chains:
        chain_length = len(chain)
        
        if chain_length == max_chain_length:
            # Full chain - check for branching
            has_branching = False
            for node in chain:
                in_degree = tracker.G.in_degree(node)
                out_degree = tracker.G.out_degree(node)
                if in_degree > 1 or out_degree > 1:
                    has_branching = True
                    break
            
            if has_branching:
                full_chains_branching.append(chain)
            else:
                full_chains_width_1.append(chain)
        else:
            # Partial chains
            if chain_length >= 3:
                partial_chains_long.append(chain)
            elif chain_length == 2:
                partial_chains_short.append(chain)
    
    return {
        'full_width_1': full_chains_width_1,
        'full_branching': full_chains_branching,
        'partial_long': partial_chains_long,
        'partial_short': partial_chains_short,
        'singleton': [c for c in all_chains if len(c) == 1]
    }


print("✓ Visualization functions loaded")

## 4. Initialize and Load Data

Initialize the tracker and load crown data with image extraction.

In [ ]:
# Initialize the tracker
tracker = TreeTrackingGraph(
    crown_dir='../../output/input_crowns',
    ortho_dir='../../input/input_om',
    output_dir='../../output',
    auto_discover=True
)

print(f"\nDiscovered {len(tracker.file_pairs)} orthomosaic pairs")
print(f"OM IDs: {tracker.om_ids}")
print("\nFile pairs:")
for om_id, (crown_file, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
    print(f"  OM{om_id}: {os.path.basename(crown_file)}, {os.path.basename(ortho_file) if ortho_file else 'N/A'}")

In [ ]:
# Load crown data and extract RGB images from ORIGINAL (unshifted) geometries
# CRITICAL: Images are extracted BEFORE any spatial alignment
tracker.load_data(load_images=True)

print(f"\n✓ Loaded data from {len(tracker.om_ids)} OMs")
print(f"✓ Extracted images: {sum(sum(img is not None for img in imgs) for imgs in tracker.crown_images.values())} total")

## 5. Apply Spatial Alignment

Apply consecutive alignment using nearest neighbors and median translation.

In [ ]:
# Apply consecutive alignment
# Note: Images remain from ORIGINAL geometries, only crown polygons are shifted
tracker.load_data(load_images=True, align=True)

print("\nAlignment Shifts Summary:")
print("="*60)
for om_id, (dx, dy) in tracker.alignment_shifts.items():
    mag = np.sqrt(dx**2 + dy**2)
    print(f"OM{om_id}: dx={dx:>7.2f}m, dy={dy:>7.2f}m, magnitude={mag:>7.2f}m")
print("="*60)

## 6. Build Tracking Graph

Build the crown tracking graph with ultra-relaxed parameters.

In [ ]:
# Build the tracking graph
tracker.build_graph_conditional()

# Generate quality report
report, metrics = tracker.quality_report()
print(report)

# Save report
tracker.save_text(report, 'crown_tracking_quality_report.txt')
tracker.save_json(metrics, 'crown_tracking_metrics.json')

In [ ]:
# # Example: Visualize a specific tracking chain
# example_chain = tracker.get_matching_chain(start_om_id=1, crown_id=0)
# if len(example_chain) > 1:
#     visualize_chain_with_extracted_images(
#         example_chain, 
#         tracker.crowns_gdfs, 
#         tracker.crown_images, 
#         tracker, 
#         title="Example Tracking Chain"
#     )
# else:
#     print("No chain found for OM1, Crown 0")

## 7. Analyze Tracking Results

Categorize and analyze the tracking chains.

In [ ]:
# Categorize chains
chain_categories = categorize_chains(tracker)

print("Chain Analysis:")
print(f"Full chains (no branching): {len(chain_categories['full_width_1'])}")
print(f"Full chains (with branching): {len(chain_categories['full_branching'])}")
print(f"Partial chains (long): {len(chain_categories['partial_long'])}")
print(f"Partial chains (short): {len(chain_categories['partial_short'])}")
print(f"Singletons: {len(chain_categories['singleton'])}")

# Show example of each category
for category, chains in chain_categories.items():
    if chains:
        print(f"\nExample {category} chain: {chains[0]}")

In [ ]:
# # Visualize all 5 crown sets: aligned vs non-aligned for comparison
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# # Load original non-aligned crowns for comparison
# tracker_original = TreeTrackingGraph(
#     crown_dir='../../output/input_crowns',
#     ortho_dir='../../input/input_om',
#     output_dir='../../output',
#     auto_discover=True
# )
# tracker_original.load_data(load_images=False, align=False)  # No alignment

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 12))

# colors = ['red', 'blue', 'green', 'orange', 'purple']

# # Plot 1: Non-aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker_original.om_ids):
#     gdf = tracker_original.crowns_gdfs[om_id]
#     gdf.plot(ax=ax1, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax1.set_title('Crown Sets Before Spatial Alignment (Original Positions)')
# ax1.legend(handles=legend_handles)
# ax1.set_aspect('equal')
# ax1.set_xlabel('X (m)')
# ax1.set_ylabel('Y (m)')
# ax1.grid(True, alpha=0.3)

# # Plot 2: Aligned crowns
# legend_handles = []
# for i, om_id in enumerate(tracker.om_ids):
#     gdf = tracker.crowns_gdfs[om_id]
#     gdf.plot(ax=ax2, facecolor='none', edgecolor=colors[i], linewidth=2, alpha=0.8)
#     legend_handles.append(mpatches.Patch(edgecolor=colors[i], facecolor='none', linewidth=2, label=f'OM{om_id}'))

# ax2.set_title('Crown Sets After Spatial Alignment (Corrected Positions)')
# ax2.legend(handles=legend_handles)
# ax2.set_aspect('equal')
# ax2.set_xlabel('X (m)')
# ax2.set_ylabel('Y (m)')
# ax2.grid(True, alpha=0.3)

# plt.tight_layout()
# plt.show()

# print("Alignment comparison: Left shows original positions, right shows after spatial corrections.")
# print("If alignment worked, crowns should be better aligned spatially on the right.")

In [ ]:
# Visualize all complete 5-length tracking chains
print("Visualizing all complete 5-length tracking chains...")

# Get all full chains (both no-branching and with-branching)
full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']

print(f"Found {len(full_chains)} complete chains to visualize")

for i, chain in enumerate(full_chains):
    chain_type = "No Branching" if chain in chain_categories['full_width_1'] else "With Branching"
    title = f"Complete Chain {i+1}/{len(full_chains)} - {chain_type}"

    visualize_chain_with_extracted_images(
        chain,
        tracker.crowns_gdfs,
        tracker.crown_images,
        tracker,
        title=title
    )

print(f"✓ Completed visualization of all {len(full_chains)} complete tracking chains")

## 8. PPT-friendly chain sheets
Create 3 large images (6 chains each) so they can be dropped into slides easily.

In [ ]:
from math import ceil

def save_full_chain_panels(chains, tracker, output_dir=None, dpi=150, show=False):
    output_dir = Path(output_dir or Path(tracker.output_dir) / "full_chain_panels")
    output_dir.mkdir(parents=True, exist_ok=True)
    saved_paths = []
    for idx, chain in enumerate(chains, 1):
        path = output_dir / f"chain_{idx:02d}.png"
        visualize_chain_with_extracted_images(
            chain,
            tracker.crowns_gdfs,
            tracker.crown_images,
            tracker,
            title=f"Chain {idx} (length {len(chain)})",
            save_path=path,
            show=show,
            close_fig=True,
            dpi=dpi,
        )
        saved_paths.append(path)
    return saved_paths


In [ ]:

def build_chain_collages(image_paths, chains_per_sheet=6, cols=2, dpi=200, output_dir=None):
    output_dir = Path(output_dir or Path(tracker.output_dir) / "ppt_collages")
    output_dir.mkdir(parents=True, exist_ok=True)
    sheets = []
    for sheet_start in range(0, len(image_paths), chains_per_sheet):
        chunk = image_paths[sheet_start:sheet_start + chains_per_sheet]
        rows = ceil(len(chunk) / cols) if chunk else 1
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 12, rows * 12))
        axes_arr = np.atleast_1d(axes).ravel()
        for ax_idx, ax in enumerate(axes_arr):
            if ax_idx >= len(chunk):
                ax.axis('off')
                continue
            img = plt.imread(chunk[ax_idx])
            ax.imshow(img)
            ax.set_title(chunk[ax_idx].name, fontsize=12)
            ax.axis('off')
        fig.tight_layout()
        sheet_path = output_dir / f"chains_sheet_{(sheet_start // chains_per_sheet) + 1:02d}.png"
        fig.savefig(sheet_path, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        sheets.append(sheet_path)
    return sheets

# Generate PPT-friendly sheets for all complete chains (6 per sheet, expect 3 sheets for 18 chains)
full_chains = chain_categories['full_width_1'] + chain_categories['full_branching']
print(f"Preparing PPT sheets for {len(full_chains)} full chains...")
chain_panel_paths = save_full_chain_panels(full_chains, tracker, dpi=150, show=False)
sheet_paths = build_chain_collages(chain_panel_paths, chains_per_sheet=6, cols=3, dpi=200)
print("Sheets saved:")
for p in sheet_paths:
    print(f" - {p}")